# Compute the caption text embeddings for the CC12M dataset

Source:

   https://github.com/camlaedtke/imagen_pytorch

In [ ]:
import json
import torch
import random
import braceexpand
from   time        import time
from   tqdm        import tqdm
import webdataset  as     wds
from   source.t5   import t5_encode_text

In [ ]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
# device = "cpu"
print(f'[INFO] The computing device is {device}')

In [ ]:
'''
Get the text embedding tensor for a given text using T5 model.
'''
def get_emb_tensor(text):
    text_embeds = t5_encode_text([text], name="google/t5-v1_1-xl", return_attn_mask=False)
    return text_embeds.cpu()

'''
Get the number of successfully downloaded images (given the URLs) from 
the stats JSON file associated to the input TAR file.
'''
def get_count(input_file):
    stats_file = input_file[:-4] + "_stats.json"
    f          = open(stats_file)
    stats      = json.load(f)
    f.close()
    count      = stats["successes"]
    return count

'''
Get a batch of text embeddings for a list of texts using the T5 model.
'''
def get_emb_batch(text):
    text_embeds = t5_encode_text(text, name="google/t5-v1_1-xl", return_attn_mask=False)
    text_embeds = text_embeds.cpu()
    emb_batch   = []
    for tensor in text_embeds:
        idx, idy       = tensor.nonzero(as_tuple=True)
        tensor_nonzero = tensor[None, 0:max(idx)+1:]
        emb_batch.append(tensor_nonzero)
    return emb_batch

In [ ]:
'''
Augment a WebDataset TAR file by adding T5 text embeddings for each caption.
The output TAR file will contain the original image (as a PNG file), the caption 
(as a TXT file), and the text embedding (with an extension ".emb.pyd").
''' 
def batch_augment_wds(input_shard, output_shard, batch_size):
    start       = time()
    count       = get_count(input_shard)
    input_shard = "file:" + input_shard
    
    src = wds.DataPipeline(
        wds.SimpleShardList(input_shard),
        wds.tarfile_to_samples(),
        wds.decode("pil"),
        wds.to_tuple("__key__", "jpg;png", "txt")
    )
    
    idx       = 1
    batch_idx = 0
    keys      = []
    imgs      = []
    caps      = []
    embs      = []
    for key, img, cap in tqdm(src, total=count, desc=f"Extracting {input_shard}"):
        keys.append(key)
        imgs.append(img)
        caps.append(cap)
        
        if ((idx%batch_size) == 0 and idx != 1) or idx == count:
            emb_batch = get_emb_batch(caps[batch_size*batch_idx:])
            for emb in emb_batch:
                embs.append(emb)
            batch_idx += 1
        idx += 1
                
    dst = wds.TarWriter(output_shard)
    for key, img, cap, emb in tqdm(zip(keys, imgs, caps, embs), total=count, desc=f"Writing {output_shard}"):
        dst.write({
            "__key__": key, 
            "png":     img, 
            "txt":     cap, 
            "emb.pyd": emb
        })
        
    end = time()
    print(f"Finished - processing took {(end-start)/60:.0f}m : {(end-start)%60:.0f}s")

In [ ]:
'''
Shuffle and augment a WebDataset TAR file by adding T5 text embeddings for each caption.
The output TAR file will contain the original image (as PNG file), the caption (as TXT file),
and the text embedding (with an extension ".emb.pyd").
'''
def shuffle_augment_wds(input, output):
    """
    Takes ~450s to process each TAR file.
    """
    start = time()
    count = get_count(input) # number of TAR files
    input = "file:" + input  # prepend "file:" to each input file path
    
    # create a WebDataset processing pipeline
    src   = wds.DataPipeline(
        # creates an iterator over all the input TARs
        wds.SimpleShardList(input),
        # creates an iterator over the TARs assigned to each worker
        wds.tarfile_to_samples(),
        # decodes the images
        wds.decode("pil"),
        # Creates a tuple with the key, image, caption, and caption of each sample
        wds.to_tuple("__key__", "jpg;png", "txt", "txt"),
        # applies the get_emb_tensor() function to the 4th element (the repeated caption) 
        # of each tuple to obtain the text embedding tensor
        wds.map_tuple(None, None, None, get_emb_tensor)
    )
    
    # collect all samples in a list and shuffle them
    samples = []
    for key, img, cap, emb in tqdm(src, total=count, desc=f"Extracting content from {input}"):
        samples.append([key, img, cap, emb])
    random.shuffle(samples)    
    
    # write the shuffled samples to the output TAR file
    dst = wds.TarWriter(output)
    for sample in tqdm(samples, total=count, desc=f"Writing new content to  {output}"):
        dst.write({
            "__key__": sample[0], 
            "png":     sample[1], 
            "txt":     sample[2], 
            "emb.pyd": sample[3]
        })
    end = time()
    print(f"Finished - processing took {(end-start)/60:.0f}m : {(end-start)%60:.0f}s")

In [ ]:
'''
FIRST EXAMPLE OF USE:

For each input TAR file in the specified range, augment it by adding T5 text embeddings for each caption.
The output TAR files will contain the original image (as PNG file), the caption (as TXT file),
and the text embedding (as PYD file).
'''
#input_shards  = braceexpand.braceexpand("cc12m_original/{00000..01242}.tar")
#output_shards = braceexpand.braceexpand("file:OUR_DATASETS_DIR/cc12m_embeds/{00000..01242}.tar")
#for input_shard, output_shard in zip(input_shards, output_shards):
#     batch_augment_wds(input_shard, output_shard, batch_size=8)

In [ ]:
'''
SECOND EXAMPLE OF USE:

For each input TAR file in the specified range, shuffle and augment it 
by adding T5 text embeddings for each caption. The output TAR files will 
contain the original image (as PNG file), the caption (as TXT file), and 
the text embedding (as PYD file).
'''
input_shards  = braceexpand.braceexpand("OUR_DATASETS_DIR/cc12m/{00000..01242}.tar")
output_shards = braceexpand.braceexpand("file:OUR_DATASETS_DIR/cc12m_embeds/{00000..01242}.tar")
for input_shard,  output_shard in zip(input_shards, output_shards):
    shuffle_augment_wds(input=input_shard, output=output_shard)